In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

processed_dir_2018 = Path("../data/processed/cicids18")

processed_files_2018 = sorted(
    processed_dir_2018.glob("*.parquet")
)

print("CSE-CIC-IDS2018 processed files:", len(processed_files_2018))

for file_path in processed_files_2018:
    print(file_path.name)

CSE-CIC-IDS2018 processed files: 10
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


In [3]:
df_2018 = pd.concat(
    [pd.read_parquet(file_path) for file_path in processed_files_2018],
    ignore_index=True
)

print("Shape:", df_2018.shape)
print("Columns:", len(df_2018.columns))
print()
print(df_2018.head())

Shape: (6659532, 71)
Columns: 71

   Flow Duration  Total Fwd Packets  Total Backward Packets  \
0         141385                  9                       7   
1            281                  2                       1   
2         279824                 11                      15   
3            132                  2                       0   
4         274016                  9                      13   

   Fwd Packets Length Total  Bwd Packets Length Total  Fwd Packet Length Max  \
0                       553                    3773.0                    202   
1                        38                       0.0                     38   
2                      1086                   10527.0                    385   
3                         0                       0.0                      0   
4                      1285                    6141.0                    517   

   Fwd Packet Length Min  Fwd Packet Length Mean  Fwd Packet Length Std  \
0                      0       

In [3]:
print("Target values:")
print(df_2018["Target"].value_counts())

print()
print("Missing values:", df_2018.isna().sum().sum())

print()
print("Infinite values:", np.isinf(
    df_2018.select_dtypes(include=np.number)
).sum().sum())

print()
print("Unique source files:", df_2018["Source_File"].nunique())

print()
print("Original labels:")
print(df_2018["Original_Label"].value_counts())

Target values:
Target
0    5329008
1    1330524
Name: count, dtype: int64

Missing values: 0

Infinite values: 0

Unique source files: 10

Original labels:
Original_Label
Benign                      5329008
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


In [4]:
feature_columns = [
    column for column in df_2018.columns
    if column not in ["Original_Label", "Target", "Source_File"]
]

print("Number of modeling features:", len(feature_columns))
print()
print(feature_columns)

Number of modeling features: 68

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Avg Pa

In [5]:
X_2018 = df_2018[feature_columns]
y_2018 = df_2018["Target"]

print("X shape:", X_2018.shape)
print("y shape:", y_2018.shape)

print()
print("Feature columns:", len(feature_columns))
print("Unique feature names:", len(set(feature_columns)))

print()
print("Non-numeric features:")
print(
    X_2018.select_dtypes(exclude=np.number).columns.tolist()
)

X shape: (6659532, 68)
y shape: (6659532,)

Feature columns: 68
Unique feature names: 68

Non-numeric features:
[]


In [9]:
feature_hashes_2018 = pd.util.hash_pandas_object(
    X_2018.astype(np.float64),
    index=False
)

print("Total fingerprints:", len(feature_hashes_2018))
print("Unique fingerprints:", feature_hashes_2018.nunique())
print(
    "Rows belonging to duplicate groups:",
    len(feature_hashes_2018) - feature_hashes_2018.nunique()
)

Total fingerprints: 6659532
Unique fingerprints: 6297984
Rows belonging to duplicate groups: 361548


In [10]:
duplicate_mask_2018 = feature_hashes_2018.duplicated(keep=False)

duplicate_groups_2018 = (
    df_2018.loc[duplicate_mask_2018]
    .assign(Row_Hash=feature_hashes_2018[duplicate_mask_2018].values)
    .groupby("Row_Hash")["Target"]
    .agg(["nunique", "count"])
)

conflicting_duplicate_groups_2018 = duplicate_groups_2018[
    duplicate_groups_2018["nunique"] > 1
]

print(
    "Duplicate fingerprint groups:",
    len(duplicate_groups_2018)
)

print(
    "Conflicting duplicate fingerprint groups:",
    len(conflicting_duplicate_groups_2018)
)

print(
    "Rows belonging to conflicting groups:",
    duplicate_mask_2018[
        feature_hashes_2018.isin(
            set(conflicting_duplicate_groups_2018.index)
        )
    ].sum()
)

Duplicate fingerprint groups: 200740
Conflicting duplicate fingerprint groups: 21947
Rows belonging to conflicting groups: 70102


In [11]:
conflicting_hash_values_2018 = set(
    conflicting_duplicate_groups_2018.index
)

conflicting_group_mask_2018 = feature_hashes_2018.isin(
    conflicting_hash_values_2018
)

print(
    df_2018.loc[
        conflicting_group_mask_2018,
        "Target"
    ].value_counts()
)

print()

print(
    df_2018.loc[
        conflicting_group_mask_2018,
        "Original_Label"
    ].value_counts().head(20)
)

Target
0    44005
1    26097
Name: count, dtype: int64

Original_Label
Benign                   44005
Infilteration            25841
Bot                        197
DoS attacks-Slowloris       52
Brute Force -XSS             3
Brute Force -Web             3
SQL Injection                1
Name: count, dtype: int64


In [12]:
conflicting_by_file_2018 = (
    df_2018.loc[
        conflicting_group_mask_2018,
        ["Source_File", "Target"]
    ]
    .groupby("Source_File")
    .agg(
        Rows=("Target", "size"),
        Attack_Rows=("Target", "sum")
    )
)

conflicting_by_file_2018["Attack_Rate"] = (
    conflicting_by_file_2018["Attack_Rows"]
    / conflicting_by_file_2018["Rows"]
)

conflicting_by_file_2018.sort_values(
    "Rows",
    ascending=False
)

,Rows,Attack_Rows,Attack_Rate
Source_File,,,
Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet,25718,12895,0.501400
Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet,25256,12946,0.512591
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet,14979,197,0.013152
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet,1016,52,0.051181
Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet,833,1,0.001200
Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet,832,6,0.007212
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet,744,0,0.000000
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet,718,0,0.000000
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet,6,0,0.000000


In [13]:
df_2018_clean = df_2018.loc[
    ~conflicting_group_mask_2018
].copy()

print("Retained dataset:")
print("Rows:", len(df_2018))
print(
    "Attack rate:",
    df_2018["Target"].mean()
)

print()

print("Conflict-removed dataset:")
print("Rows:", len(df_2018_clean))
print(
    "Attack rate:",
    df_2018_clean["Target"].mean()
)

print()

print("Rows removed:", len(df_2018) - len(df_2018_clean))

Retained dataset:
Rows: 6659532
Attack rate: 0.19979241784557833

Conflict-removed dataset:
Rows: 6589430
Attack rate: 0.19795748645937508

Rows removed: 70102


In [11]:
clean_hashes_2018 = pd.util.hash_pandas_object(
    df_2018_clean[feature_columns].astype(np.float64),
    index=False
)

clean_label_check_2018 = (
    pd.DataFrame({
        "Row_Hash": clean_hashes_2018,
        "Target": df_2018_clean["Target"].to_numpy()
    })
    .groupby("Row_Hash")["Target"]
    .nunique()
)

remaining_conflicts_2018 = (
    clean_label_check_2018 > 1
).sum()

print(
    "Remaining conflicting fingerprint groups:",
    remaining_conflicts_2018
)

Remaining conflicting fingerprint groups: 0


In [12]:
from sklearn.model_selection import StratifiedGroupKFold

groups_2018 = clean_hashes_2018

splitter_2018 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx_2018, test_idx_2018 = next(
    splitter_2018.split(
        df_2018_clean[feature_columns],
        df_2018_clean["Target"],
        groups=groups_2018
    )
)

train_groups_2018 = set(
    groups_2018.iloc[train_idx_2018]
)

test_groups_2018 = set(
    groups_2018.iloc[test_idx_2018]
)

print("Train rows:", len(train_idx_2018))
print("Test rows:", len(test_idx_2018))

print()
print(
    "Test proportion:",
    len(test_idx_2018) / len(df_2018_clean)
)

print()
print(
    "Overlapping fingerprint groups:",
    len(train_groups_2018.intersection(test_groups_2018))
)

Train rows: 5271544
Test rows: 1317886

Test proportion: 0.2

Overlapping fingerprint groups: 0


In [13]:
y_clean_2018 = df_2018_clean["Target"]

print("Overall attack rate:", y_clean_2018.mean())
print(
    "Train attack rate:",
    y_clean_2018.iloc[train_idx_2018].mean()
)
print(
    "Test attack rate:",
    y_clean_2018.iloc[test_idx_2018].mean()
)

print()
print("Train class counts:")
print(
    y_clean_2018.iloc[train_idx_2018].value_counts()
    .sort_index()
)

print()
print("Test class counts:")
print(
    y_clean_2018.iloc[test_idx_2018].value_counts()
    .sort_index()
)

Overall attack rate: 0.19795748645937508
Train attack rate: 0.19795737264072916
Test attack rate: 0.19795794173395878

Train class counts:
Target
0    4228003
1    1043541
Name: count, dtype: int64

Test class counts:
Target
0    1057000
1     260886
Name: count, dtype: int64


In [14]:
split_dir = Path("../outputs/splits")
split_dir.mkdir(parents=True, exist_ok=True)

np.save(
    split_dir / "2018_train_idx.npy",
    train_idx_2018
)

np.save(
    split_dir / "2018_test_idx.npy",
    test_idx_2018
)

print("2018 split indices saved.")

2018 split indices saved.


In [34]:
from pathlib import Path
import numpy as np

split_dir = Path("../outputs/splits")

np.save(
    split_dir / "feature_columns_2018.npy",
    np.array(feature_columns, dtype=object)
)

print("2018 feature list saved.")

2018 feature list saved.


In [15]:
X_train_2018 = df_2018_clean[feature_columns].iloc[train_idx_2018]
X_test_2018 = df_2018_clean[feature_columns].iloc[test_idx_2018]

y_train_2018 = df_2018_clean["Target"].iloc[train_idx_2018]
y_test_2018 = df_2018_clean["Target"].iloc[test_idx_2018]

print("X_train:", X_train_2018.shape)
print("X_test:", X_test_2018.shape)
print("y_train:", y_train_2018.shape)
print("y_test:", y_test_2018.shape)

X_train: (5271544, 68)
X_test: (1317886, 68)
y_train: (5271544,)
y_test: (1317886,)


In [16]:
print("Training missing values:", X_train_2018.isna().sum().sum())
print("Test missing values:", X_test_2018.isna().sum().sum())

print()
print("Training target values:")
print(y_train_2018.value_counts().sort_index())

print()
print("Test target values:")
print(y_test_2018.value_counts().sort_index())

Training missing values: 0
Test missing values: 0

Training target values:
Target
0    4228003
1    1043541
Name: count, dtype: int64

Test target values:
Target
0    1057000
1     260886
Name: count, dtype: int64


In [17]:
rf_pipeline_2018 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [18]:
rf_pipeline_2018.fit(
    X_train_2018,
    y_train_2018
)

print("Random Forest 2018 training completed.")

Random Forest 2018 training completed.


In [19]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    }

    return results, confusion_matrix(y_test, y_pred)

In [20]:
rf_results_2018, rf_cm_2018 = evaluate_model(
    rf_pipeline_2018,
    X_test_2018,
    y_test_2018
)

print("Random Forest 2018 Results:")
for metric, value in rf_results_2018.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion Matrix:")
print(rf_cm_2018)

Random Forest 2018 Results:
Accuracy: 0.978706
Precision: 0.957727
Recall: 0.933642
F1: 0.945531
ROC-AUC: 0.985733
PR-AUC: 0.975644

Confusion Matrix:
[[1046249   10751]
 [  17312  243574]]


In [21]:
xgb_params_2018 = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

xgb_model_2018 = xgb.XGBClassifier(
    **xgb_params_2018
)

In [22]:
xgb_model_2018.fit(
    X_train_2018,
    y_train_2018
)

print("XGBoost 2018 training completed.")

XGBoost 2018 training completed.


In [23]:
xgb_results_2018, xgb_cm_2018 = evaluate_model(
    xgb_model_2018,
    X_test_2018,
    y_test_2018
)

print("XGBoost 2018 Results:")
for metric, value in xgb_results_2018.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion Matrix:")
print(xgb_cm_2018)

XGBoost 2018 Results:
Accuracy: 0.985830
Precision: 0.996780
Recall: 0.931426
F1: 0.962995
ROC-AUC: 0.992903
PR-AUC: 0.983759

Confusion Matrix:
[[1056215     785]
 [  17890  242996]]


In [24]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier

tuning_model_2018 = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

param_distributions_2018 = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [1.0, 2.0, 4.0]
}

cv_strategy_2018 = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search_2018 = RandomizedSearchCV(
    estimator=tuning_model_2018,
    param_distributions=param_distributions_2018,
    n_iter=12,
    scoring="average_precision",
    cv=cv_strategy_2018,
    verbose=2,
    random_state=42,
    n_jobs=1,
    return_train_score=True
)

In [25]:
import time

start_time = time.time()

random_search_2018.fit(
    X_train_2018,
    y_train_2018
)

elapsed_time = time.time() - start_time

print()
print("Tuning completed.")
print(f"Time: {elapsed_time / 60:.2f} minutes")
print()
print("Best parameters:")
print(random_search_2018.best_params_)
print()
print("Best CV PR-AUC:")
print(random_search_2018.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  32.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  35.2s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=8, min_child_weight=10, n_estimators=100, scale_pos_weight=4.0, subsample=1.0; total time=  41.7s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  56.4s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  44.4s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, min_child_weight=1, n_estimators=100, scale_pos_weight=4.0, subsample=0.8; total time=  

In [26]:
best_params_2018 = random_search_2018.best_params_

xgb_tuned_2018 = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **best_params_2018
)

In [27]:
xgb_tuned_2018.fit(
    X_train_2018,
    y_train_2018
)

print("Final tuned 2018 XGBoost training completed.")

Final tuned 2018 XGBoost training completed.


In [28]:
xgb_tuned_results_2018, xgb_tuned_cm_2018 = evaluate_model(
    xgb_tuned_2018,
    X_test_2018,
    y_test_2018
)

print("Tuned XGBoost 2018 Results:")
for metric, value in xgb_tuned_results_2018.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion Matrix:")
print(xgb_tuned_cm_2018)

Tuned XGBoost 2018 Results:
Accuracy: 0.983154
Precision: 0.972256
Recall: 0.941775
F1: 0.956773
ROC-AUC: 0.993717
PR-AUC: 0.985012

Confusion Matrix:
[[1049989    7011]
 [  15190  245696]]


In [ ]:
processed_dir_2017 = Path("../data/processed/cicids17")
processed_files_2017 = sorted(processed_dir_2017.glob("*.parquet"))

df_2017_external = pd.concat(
    [pd.read_parquet(file_path) for file_path in processed_files_2017],
    ignore_index=True
)

feature_columns_2017 = [
    column for column in df_2017_external.columns
    if column not in ["Original_Label", "Target", "Source_File"]
]

assert len(feature_columns_2017) == 68
assert feature_columns_2017 == feature_columns

X_2017_external = df_2017_external[feature_columns_2017]
y_2017_external = df_2017_external["Target"]

print("2017 external dataset shape:", X_2017_external.shape)
print("2017 external target shape:", y_2017_external.shape)
print("2017 attack rate:", y_2017_external.mean())
print("Missing values:", X_2017_external.isna().sum().sum())

2017 external dataset shape: (2830743, 68)
2017 external target shape: (2830743,)
2017 attack rate: 0.19699633629757277
Missing values: 5734


In [30]:
feature_hashes_2017 = pd.util.hash_pandas_object(
    X_2017_external.astype(np.float64),
    index=False
)

duplicate_counts_2017 = feature_hashes_2017.value_counts()

duplicate_hashes_2017 = duplicate_counts_2017[
    duplicate_counts_2017 > 1
].index

duplicate_mask_2017 = feature_hashes_2017.isin(duplicate_hashes_2017)

duplicate_labels_2017 = pd.DataFrame({
    "Hash": feature_hashes_2017[duplicate_mask_2017].values,
    "Target": y_2017_external[duplicate_mask_2017].values
})

conflicting_duplicate_groups_2017 = (
    duplicate_labels_2017
    .groupby("Hash")["Target"]
    .nunique()
)

conflicting_hash_values_2017 = set(
    conflicting_duplicate_groups_2017[
        conflicting_duplicate_groups_2017 > 1
    ].index
)

conflicting_group_mask_2017 = feature_hashes_2017.isin(
    conflicting_hash_values_2017
)

X_2017_external = X_2017_external.loc[
    ~conflicting_group_mask_2017
].copy()

y_2017_external = y_2017_external.loc[
    ~conflicting_group_mask_2017
].copy()

remaining_hashes_2017 = pd.util.hash_pandas_object(
    X_2017_external.astype(np.float64),
    index=False
)

remaining_labels_2017 = pd.DataFrame({
    "Hash": remaining_hashes_2017.values,
    "Target": y_2017_external.values
})

remaining_conflicts_2017 = (
    remaining_labels_2017
    .groupby("Hash")["Target"]
    .nunique()
)

print("Clean 2017 external dataset shape:", X_2017_external.shape)
print("Clean 2017 target shape:", y_2017_external.shape)
print("Clean 2017 attack rate:", y_2017_external.mean())
print("Missing values:", X_2017_external.isna().sum().sum())
print(
    "Remaining conflicting groups:",
    (remaining_conflicts_2017 > 1).sum()
)

Clean 2017 external dataset shape: (2747873, 68)
Clean 2017 target shape: (2747873,)
Clean 2017 attack rate: 0.17321033395648197
Missing values: 4910
Remaining conflicting groups: 0


In [31]:
y_pred_2017_external = xgb_tuned_2018.predict(X_2017_external)
y_prob_2017_external = xgb_tuned_2018.predict_proba(X_2017_external)[:, 1]

results_2018_to_2017 = {
    "Accuracy": accuracy_score(y_2017_external, y_pred_2017_external),
    "Precision": precision_score(y_2017_external, y_pred_2017_external, zero_division=0),
    "Recall": recall_score(y_2017_external, y_pred_2017_external, zero_division=0),
    "F1": f1_score(y_2017_external, y_pred_2017_external, zero_division=0),
    "ROC-AUC": roc_auc_score(y_2017_external, y_prob_2017_external),
    "PR-AUC": average_precision_score(y_2017_external, y_prob_2017_external)
}

cm_2018_to_2017 = confusion_matrix(
    y_2017_external,
    y_pred_2017_external
)

print("2018 → 2017 External Transfer Results:")

for metric, value in results_2018_to_2017.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion Matrix:")
print(cm_2018_to_2017)

2018 → 2017 External Transfer Results:
Accuracy: 0.788528
Precision: 0.339926
Recall: 0.234545
F1: 0.277570
ROC-AUC: 0.662023
PR-AUC: 0.269757

Confusion Matrix:
[[2055140  216773]
 [ 364326  111634]]


Bi-directional Results comparison

In [32]:
comparison_df = pd.DataFrame([
    {
        "Direction": "2017 → 2017",
        "Evaluation": "Within-dataset",
        "Accuracy": 0.999029,
        "Precision": 0.996590,
        "Recall": 0.998485,
        "F1": 0.997537,
        "ROC-AUC": 0.999973,
        "PR-AUC": 0.999878
    },
    {
        "Direction": "2017 → 2018",
        "Evaluation": "Cross-dataset",
        "Accuracy": 0.801322,
        "Precision": 0.866640,
        "Recall": 0.006594,
        "F1": 0.013088,
        "ROC-AUC": 0.778893,
        "PR-AUC": 0.574965
    },
    {
        "Direction": "2018 → 2018",
        "Evaluation": "Within-dataset",
        "Accuracy": 0.983154,
        "Precision": 0.972256,
        "Recall": 0.941775,
        "F1": 0.956773,
        "ROC-AUC": 0.993717,
        "PR-AUC": 0.985012
    },
    {
        "Direction": "2018 → 2017",
        "Evaluation": "Cross-dataset",
        "Accuracy": 0.788528,
        "Precision": 0.339926,
        "Recall": 0.234545,
        "F1": 0.277570,
        "ROC-AUC": 0.662023,
        "PR-AUC": 0.269757
    }
])

comparison_df

,Direction,Evaluation,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,2017 → 2017,Within-dataset,0.999029,0.996590,0.998485,0.997537,0.999973,0.999878
1,2017 → 2018,Cross-dataset,0.801322,0.866640,0.006594,0.013088,0.778893,0.574965
2,2018 → 2018,Within-dataset,0.983154,0.972256,0.941775,0.956773,0.993717,0.985012
3,2018 → 2017,Cross-dataset,0.788528,0.339926,0.234545,0.277570,0.662023,0.269757


In [33]:
degradation_df = pd.DataFrame([
    {
        "Direction": "2017 → 2018",
        "Recall Drop": 0.998485 - 0.006594,
        "F1 Drop": 0.997537 - 0.013088,
        "ROC-AUC Drop": 0.999973 - 0.778893,
        "PR-AUC Drop": 0.999878 - 0.574965
    },
    {
        "Direction": "2018 → 2017",
        "Recall Drop": 0.941775 - 0.234545,
        "F1 Drop": 0.956773 - 0.277570,
        "ROC-AUC Drop": 0.993717 - 0.662023,
        "PR-AUC Drop": 0.985012 - 0.269757
    }
])

degradation_df

,Direction,Recall Drop,F1 Drop,ROC-AUC Drop,PR-AUC Drop
0,2017 → 2018,0.991891,0.984449,0.221080,0.424913
1,2018 → 2017,0.707230,0.679203,0.331694,0.715255


In [35]:
from pathlib import Path

model_dir = Path("../outputs/models")
model_dir.mkdir(parents=True, exist_ok=True)

xgb_tuned_2018.save_model(
    model_dir / "xgb_tuned_2018.json"
)

print("2018 tuned XGBoost model saved.")

2018 tuned XGBoost model saved.


In [14]:
import numpy as np
import xgboost as xgb

xgb_tuned_2018 = xgb.XGBClassifier()
xgb_tuned_2018.load_model("../outputs/models/xgb_tuned_2018.json")

feature_columns_2018 = np.load("../outputs/splits/feature_columns_2018.npy", allow_pickle=True).tolist()

print("Model loaded:", type(xgb_tuned_2018))
print("Number of features:", len(feature_columns_2018))
print("First 10 features:", feature_columns_2018[:10])

Model loaded: <class 'xgboost.sklearn.XGBClassifier'>
Number of features: 68
First 10 features: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max']


In [18]:
cic_2017_files = sorted([
    p for p in csv_files
    if p.parent.name.lower() == "cicids17"
])

print("Found:", len(cic_2017_files))
for p in cic_2017_files:
    print(p)

Found: 8
..\data\raw\cicids17\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
..\data\raw\cicids17\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
..\data\raw\cicids17\Friday-WorkingHours-Morning.pcap_ISCX.csv
..\data\raw\cicids17\Monday-WorkingHours.pcap_ISCX.csv
..\data\raw\cicids17\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
..\data\raw\cicids17\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
..\data\raw\cicids17\Tuesday-WorkingHours.pcap_ISCX.csv
..\data\raw\cicids17\Wednesday-workingHours.pcap_ISCX.csv


In [19]:
import pandas as pd
import numpy as np

cic_2017_files = sorted(cic_2017_files)

assert len(cic_2017_files) == 8, f"Expected 8 CIC-IDS2017 files, found {len(cic_2017_files)}"

dfs_2017 = []

for file in cic_2017_files:
    df_part = pd.read_csv(file)
    df_part.columns = df_part.columns.str.strip()
    dfs_2017.append(df_part)

df_2017 = pd.concat(dfs_2017, ignore_index=True)

feature_name_mapping = {
    "Average Packet Size": "Avg Packet Size",
    "Total Length of Bwd Packets": "Bwd Packets Length Total",
    "Total Length of Fwd Packets": "Fwd Packets Length Total",
    "act_data_pkt_fwd": "Fwd Act Data Packets",
    "min_seg_size_forward": "Fwd Seg Size Min",
    "Init_Win_bytes_backward": "Init Bwd Win Bytes",
    "Init_Win_bytes_forward": "Init Fwd Win Bytes",
    "Max Packet Length": "Packet Length Max",
    "Min Packet Length": "Packet Length Min"
}

df_2017 = df_2017.rename(columns=feature_name_mapping)

df_2017["target"] = (df_2017["Label"].str.strip() != "BENIGN").astype(np.int8)

missing_features = [f for f in feature_columns_2018 if f not in df_2017.columns]

assert not missing_features, f"Missing features: {missing_features}"

X_2017 = df_2017[feature_columns_2018].astype(np.float64)
y_2017 = df_2017["target"]

row_hashes_2017 = pd.util.hash_pandas_object(X_2017, index=False)

conflict_hashes_2017 = (
    pd.DataFrame({
        "hash": row_hashes_2017,
        "target": y_2017.to_numpy()
    })
    .groupby("hash")["target"]
    .nunique()
)

conflict_hashes_2017 = set(conflict_hashes_2017[conflict_hashes_2017 > 1].index)

conflict_mask_2017 = row_hashes_2017.isin(conflict_hashes_2017)

df_2017_clean = df_2017.loc[~conflict_mask_2017].copy()

X_2017_clean = df_2017_clean[feature_columns_2018].astype(np.float64)
y_2017_clean = df_2017_clean["target"]

print("Original shape:", df_2017.shape)
print("Conflicting groups:", len(conflict_hashes_2017))
print("Rows removed:", conflict_mask_2017.sum())
print("Clean shape:", df_2017_clean.shape)
print("Features:", len(feature_columns_2018))
print("Clean target distribution:")
print(y_2017_clean.value_counts())
print("Clean attack rate:", y_2017_clean.mean())

Original shape: (2830743, 80)
Conflicting groups: 272
Rows removed: 82870
Clean shape: (2747873, 80)
Features: 68
Clean target distribution:
target
0    2271913
1     475960
Name: count, dtype: int64
Clean attack rate: 0.17321033395648197


In [21]:
chunk_size = 200000
y_prob_2018_on_2017 = np.empty(len(X_2017_clean), dtype=np.float32)

for start in range(0, len(X_2017_clean), chunk_size):
    end = min(start + chunk_size, len(X_2017_clean))
    y_prob_2018_on_2017[start:end] = xgb_tuned_2018.predict_proba(
        X_2017_clean.iloc[start:end]
    )[:, 1]

print("Predictions generated:", len(y_prob_2018_on_2017))
print("Min probability:", y_prob_2018_on_2017.min())
print("Max probability:", y_prob_2018_on_2017.max())
print("Mean probability:", y_prob_2018_on_2017.mean())

Predictions generated: 2747873
Min probability: 1.0863492e-05
Max probability: 0.99917114
Mean probability: 0.17685214


In [23]:
common_families = [
    "DDoS",
    "DoS",
    "Brute Force",
    "Web Attack",
    "Bot",
    "Infiltration"
]

unmatched_2017_families = [
    "PortScan",
    "Heartbleed"
]

print("Common families:")
for family in common_families:
    print(f"- {family}")

print("\n2017-only unmatched families:")
for family in unmatched_2017_families:
    print(f"- {family}")

print("\nCounts:")
print("Common:", len(common_families))
print("2017-only:", len(unmatched_2017_families))

Common families:
- DDoS
- DoS
- Brute Force
- Web Attack
- Bot
- Infiltration

2017-only unmatched families:
- PortScan
- Heartbleed

Counts:
Common: 6
2017-only: 2


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

family_results_2018_to_2017 = []

for family in common_families + unmatched_2017_families:
    mask = df_2017_clean["attack_family"].isin(["BENIGN", family])

    y_true = (
        df_2017_clean.loc[mask, "attack_family"] == family
    ).astype(int).to_numpy()

    y_prob = y_prob_2018_on_2017[mask.to_numpy()]
    y_pred = (y_prob >= 0.5).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    family_results_2018_to_2017.append({
        "Family": family,
        "Match Type": "Common" if family in common_families else "2017-only",
        "Attack samples": int(y_true.sum()),
        "Benign": int((y_true == 0).sum()),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Predicted attacks": int(y_pred.sum()),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    })

family_results_2018_to_2017 = pd.DataFrame(family_results_2018_to_2017)

print(family_results_2018_to_2017.to_string(index=False))

      Family Match Type  Attack samples  Benign  Accuracy  Precision   Recall       F1  ROC-AUC   PR-AUC  Predicted attacks      TN     FP     FN    TP
        DDoS     Common          128021 2271913  0.856790   0.005044 0.008585 0.006355 0.435235 0.042421             217872 2055140 216773 126922  1099
         DoS     Common          247425 2271913  0.840337   0.222272 0.250391 0.235495 0.727835 0.210320             278726 2055140 216773 185472 61953
 Brute Force     Common           13835 2271913  0.899122   0.000125 0.001952 0.000234 0.381413 0.004590             216800 2055140 216773  13808    27
  Web Attack     Common            2180 2271913  0.903723   0.000051 0.005046 0.000100 0.360352 0.000727             216784 2055140 216773   2169    11
         Bot     Common            1844 2271913  0.903852   0.000000 0.000000 0.000000 0.369918 0.000594             216773 2055140 216773   1844     0
Infiltration     Common              36 2271913  0.904574   0.000023 0.138889 0.000046 0

In [25]:
output_path = "../outputs/tables/attack_family_transfer_2018_to_2017.csv"

family_results_2018_to_2017.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ../outputs/tables/attack_family_transfer_2018_to_2017.csv


Sanity audit

In [26]:
from pathlib import Path
import numpy as np

test_idx_2018 = np.load("../outputs/splits/2018_test_idx.npy")

print("Test indices:", len(test_idx_2018))
print("Minimum index:", test_idx_2018.min())
print("Maximum index:", test_idx_2018.max())

Test indices: 1317886
Minimum index: 6
Maximum index: 6589426


In [27]:
from pathlib import Path

parquet_files = list(Path("../").rglob("*.parquet"))

print("Found:", len(parquet_files))
for p in sorted(parquet_files):
    print(p)

Found: 28
..\data\processed\cicids17\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.parquet
..\data\processed\cicids17\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.parquet
..\data\processed\cicids17\Friday-WorkingHours-Morning.pcap_ISCX.parquet
..\data\processed\cicids17\Monday-WorkingHours.pcap_ISCX.parquet
..\data\processed\cicids17\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.parquet
..\data\processed\cicids17\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.parquet
..\data\processed\cicids17\Tuesday-WorkingHours.pcap_ISCX.parquet
..\data\processed\cicids17\Wednesday-workingHours.pcap_ISCX.parquet
..\data\processed\cicids18\Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
..\data\processed\cicids18\Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
..\data\processed\cicids18\DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
..\data\processed\cicids18\DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
..\data\processed\cicids